In [10]:
from conveyer import ingest, analysis
from conveyer.config import PipelineConfig
from conveyer.clustering import embed_corpus
import pandas as pd

from conveyer.models import build_embedder

In [11]:

path=""
cfg = PipelineConfig(
        data_path=path, cluster_on="question", n_clusters=12,
        embedding_backend="sentence_transformers", first_turn_only=False,
        use_llm_naming=False, out_dir=".output/",
    )


df = ingest.ingest(cfg)
df = analysis.add_brand_features(df)
df = analysis.add_intent(df, cfg)
print(f"[ingest] {df.attrs.get('source')} | rows={len(df)}")
print(f"[features] recommendation_rate={df['is_recommendation'].mean():.1%} | "
        f"brands/answer={df['n_brands_answer'].mean():.2f}")
acc = analysis.intent_accuracy(df)
if acc is not None:
    print(f"[intent] rule accuracy vs ground truth (synthetic): {acc:.1%}")

[ingest] SYNTHETIC (1500 rows;  not found) | rows=1500
[features] recommendation_rate=93.3% | brands/answer=2.36
[intent] rule accuracy vs ground truth (synthetic): 98.0%


In [12]:
mask = (df[cfg.col_session_pos] == 1) if cfg.first_turn_only else pd.Series(True, index=df.index)
work = df[mask].copy().reset_index(drop=True)
texts = work[cfg.text_column()].tolist()
embedder = build_embedder(cfg)
embeddings, emb_name = embed_corpus(texts, cfg, embedder=embedder)
print(f"[embeddings] {embeddings.shape} via {emb_name} | docs={len(texts)}")

Batches: 100%|██████████| 12/12 [00:43<00:00,  3.59s/it]

[embeddings] (1500, 1024) via sentence-transformers:BAAI/bge-large-en-v1.5 | docs=1500


In [13]:
from conveyer.clustering import *


sweep = kmeans_sweep(embeddings, cfg)
work["cluster_kmeans"] = run_kmeans(embeddings, cfg.n_clusters, cfg)
stability = kmeans_stability(embeddings, cfg.n_clusters, cfg)

bertopic_labels, _ = run_bertopic(texts, embeddings, cfg)
if bertopic_labels is not None:
    work["cluster_bertopic"] = bertopic_labels
    print(f"[bertopic] clusters={n_clusters_found(bertopic_labels)} | "
            f"noise={noise_fraction(bertopic_labels):.1%}")
else:
    print("[bertopic] unavailable -> KMeans only")

label_col = cfg.primary_label if cfg.primary_label in work.columns else "cluster_kmeans"
work["cluster"] = work[label_col]
print(f"[clustering] primary={label_col} | n={cfg.n_clusters} | kmeans stability(ARI)={stability:.3f}")

[bertopic] clusters=13 | noise=31.2%
[clustering] primary=cluster_kmeans | n=12 | kmeans stability(ARI)=0.738


In [15]:
from conveyer.analysis import *
from conveyer.models import build_llm
amp = brand_amplification(df)
summary = cluster_summary(work, cfg)
niq_scores = validate_against_niq(work, cfg)
top_reco = top_recommendations(df)
top_reco_cluster = top_recommendations_by_cluster(work)
reps = representative_docs(embeddings, work["cluster"].to_numpy(), texts)

llm = build_llm(cfg)
if llm is not None:
    names = analysis.name_clusters_llm(reps, llm)
else:
    names = analysis.name_clusters_keywords(work, texts)
work["cluster_name"] = work["cluster"].map(lambda c: names.get(int(c), {}).get("label", str(c)))

if niq_scores:
    print("[validation vs NIQ]", {k: {m: round(v, 3) for m, v in d.items()} for k, d in niq_scores.items()})

[validation vs NIQ] {'cluster_kmeans': {'ARI': 0.04, 'NMI': 0.102}, 'cluster_bertopic': {'ARI': 0.065, 'NMI': 0.236}}


In [16]:
import os

os.makedirs(cfg.out_dir, exist_ok=True)
labeled_cols = [cfg.col_question, "cluster", "cluster_kmeans", "cluster_name", "intent",
                "asks_recommendation", "is_recommendation", "n_brands_answer"]
if cfg.col_niq in work.columns:
    labeled_cols.append(cfg.col_niq)
if "cluster_bertopic" in work.columns:
    labeled_cols.append("cluster_bertopic")
labeled = work[labeled_cols]

outputs = {
    "labeled_first_turn.csv": labeled,
    "cluster_summary.csv": summary,
    "brand_amplification.csv": amp,
    "top_recommendations.csv": top_reco,
    "top_recommendations_by_cluster.csv": top_reco_cluster,
    "kmeans_sweep.csv": sweep,
}
for fname, frame in outputs.items():
    frame.to_csv(os.path.join(cfg.out_dir, fname), index=False)
print(f"[export] wrote {len(outputs)} files to {os.path.abspath(cfg.out_dir)}")

[export] wrote 6 files to c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.output
